# Figure 3: ERA5 fields behind the drag experiments

Global snapshots of ERA5 10 m wind speed, Charnock parameter, neutral minus actual wind speed, and air density.

From Özkan et al. (2025), *Sensitivity of global storm surge modelling to sea surface drag*, Ocean Dynamics, https://doi.org/10.1007/s10236-025-01713-3

The processed input files are not included in this repository; this notebook shows how the figure was made. Experiment names: in the code, `cc` = FIX, `cv` = NST, `ntr`/`uv10n` = CAD and `rho`/`uv10rho` = IFS (see the main README).

In [ ]:
# Folder with the processed input files (not included in this repository)
DATA_DIR = "path/to/paper1_data/"

In [ ]:
import os
import xarray as xr
import numpy as np
import proplot as pplt
import matplotlib.pyplot as plt
from datetime import datetime, timedelta

In [ ]:
data_dir = DATA_DIR + 'data/1-ERA5/'
year = 2008
time_range = f"{str(year-1)}12150000_{str(year+1)}01010000"

era5_msl = xr.open_dataset(data_dir + 'input_for_gtsm/ERA5_MSL_' + time_range + '.nc')
era5_c = xr.open_dataset(data_dir + 'input_for_gtsm/ERA5_C_' + time_range + '.nc')
era5_uv10 = xr.open_dataset(data_dir + 'input_for_gtsm/ERA5_UV10_' + time_range + '.nc')
era5_uv10n = xr.open_dataset(data_dir + 'input_for_gtsm/ERA5_UV10N_' + time_range + '.nc')
era5_rho = xr.open_dataset(data_dir + f'processed/AIRD/merged_ERA5_RHO_{year}.nc')

In [ ]:
# For the all datasets get the 20080101 00:00:00 data
time_step = '2008-01-01 00:00:00'
era5_msl_ts = era5_msl.sel(time=time_step)["msl"]/100
era5_c_ts = era5_c.sel(time=time_step)["c"]*1000
era5_uv10_ts = era5_uv10.sel(time=time_step)
era5_uv10n_ts = era5_uv10n.sel(time=time_step)
era5_rho_ts = era5_rho.sel(time=time_step)["rho"]

In [ ]:
# Calculate the wind speed difference
era5_uv10_ws = np.sqrt(era5_uv10_ts.u10**2 + era5_uv10_ts.v10**2)
era5_uv10n_ws = np.sqrt(era5_uv10n_ts.u10**2 + era5_uv10n_ts.v10**2)

# Calculate the wind speed difference
era5_wsdiff = era5_uv10n_ws - era5_uv10_ws

In [ ]:
pplt.rc.reso = 'xx-hi'
proj1 = pplt.Proj('robin')
fig, axs = pplt.subplots(ncols=2, nrows=2, proj=proj1, axwidth=8, share=True, span=False, panelpad='1em')

axs.format(lonlim=(-180, 180), latlim=(-80, 80),
            coast=True, coastlinewidth=0.5, lakes=False,
            rivers=False, labels=False, borders=False,
            land=True, landcolor="lightgrey",
            lonlines=20, latlines=10,
            linewidth=0.5, metacolor='black', alpha=0.5,
            abc='a', abcloc='l', abcsize=16, abcweight='bold',
            toplabelweight="normal", toplabelsize=14,
            suptitleweight="bold", suptitlesize=14,
            grid=True)

levels_msl = np.arange(900, 1100, 4)
levels_ws = np.arange(0, 21, 2)
levels_c = np.arange(0, 89, 8)
levels_wsdiff = np.arange(-0.8, 0.85, 0.1)
levels_rho = np.arange(1.120, 1.53, 0.04)

u10_shade = axs[0].contourf(era5_uv10_ws, cmap="magma", extend="max", levels=levels_ws)
msl_line = axs[0].contour(era5_msl_ts, color="black", linewidths=0.8, levels=levels_msl, zorder=1)

charnock_shade = axs[1].contourf(era5_c_ts, cmap="viridis", extend="max")
u10n_u10_shade = axs[2].contourf(era5_wsdiff, cmap="div", extend="both", levels=levels_wsdiff)
rho_shade = axs[3].contourf(era5_rho_ts, cmap="spectral", extend="both", levels=levels_rho)

lbsize = 14
cb0 = fig.colorbar(u10_shade, loc="r", label="Wind speed (m/s)", ax=axs[0], labelsize=lbsize, ticklabelsize=10)
cb1 = fig.colorbar(charnock_shade, loc="r", label="Charnock ($×10^{-3}$)", ax=axs[1], labelsize=lbsize, ticklabelsize=10)
cb2 = fig.colorbar(u10n_u10_shade, loc="r", label="Wind speed difference (m/s)", ax=axs[2], 
                   labelsize=lbsize, ticks= np.arange(-0.8, 0.85, 0.2), ticklabelsize=10)
cb3 = fig.colorbar(rho_shade, loc="r", label="Air density (kg/m³)", ax=axs[3], labelsize=lbsize, ticklabelsize=10)

ltsize = 20
axs[0].format(ltitle="Wind speed ($U_{10}$)", titlesize=ltsize)
axs[1].format(ltitle="Charnock parameter ($α_{Chvar}$)", titlesize=ltsize)
axs[2].format(ltitle="Wind speed difference ($ΔU_{10}$)", titlesize=ltsize)
axs[3].format(ltitle="Air density ($ρ_{a}$)", titlesize=ltsize)

# Save the plot as PDF and PNG, cropping the bounding box
# Define the target directory
save_dir = 'final_plots'
os.makedirs(save_dir, exist_ok=True)  # Create the directory if it doesn't exist

# Save the plot as PDF and PNG in the specified directory
fig.savefig(os.path.join(save_dir, 'global_4var.pdf'), bbox_inches='tight')
fig.savefig(os.path.join(save_dir, 'global_4var.png'), dpi=600, bbox_inches='tight')

# Show the plot
plt.show()